# AI Catcher — Pitch Recommender

Grid-search **optimal pitch type and location** for a given batter and game situation using the Expected Value engine.

**Prerequisites:** `10_expected_value_engine.ipynb` (or trained RF artifacts in `models/`)

**Physics baseline:** `data/league_average_physics.csv`

Logic lives in `src/pitch_recommender.py`.

## 1. Process

| Step | Choice |
|------|--------|
| **Spatial grid** | `plate_x` −1.5 to +1.5 ft, `plate_z` 0.5 to 4.5 ft (0.25 ft steps) → 221 locations |
| **Location features** | `is_in_zone`, `miss_dist_in`, `center_dist_in` per grid point |
| **Pitch types** | All **10 competitive** Statcast types: `CH`, `CU`, `FC`, `FF`, `FS`, `KC`, `SI`, `SL`, `ST`, `SV` |
| **Physics** | League-average speed, spin, movement, release point per pitch type |
| **Situation** | One-hot `count_state`, `base_state`, `stand`, `p_throws`, `same_handed` |
| **Batter profile** | Rolling rates + bat-tracking dict (`swing_pct` … `squared_up_rate`) — same columns as modeling frame |
| **Scoring** | Full EV chain → `expected_run_value` |
| **Smoothing** | 3-inch Gaussian spatial average → ranked **ERV** |
| **Ranking** | Sort ascending — **lowest ERV = best pitch for the pitcher** |

Output columns include **Swing Prob**, **Whiff Prob** (pitch-level), **xwOBAcon**, **ERV_raw**, and **ERV**.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

NB_DIR = Path.cwd().resolve()
ROOT = NB_DIR.parent if NB_DIR.name == "notebooks" else NB_DIR
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import importlib
import src.ev_calculator as ev_calculator
import src.pitch_recommender as pitch_recommender

importlib.reload(ev_calculator)
importlib.reload(pitch_recommender)

from src.ev_calculator import describe_ev_artifacts, load_ev_artifacts
from src.feature_engineering import league_default_batter_profile
from src.pitch_recommender import (
    CORE_PITCH_TYPES,
    RECOMMENDATION_COLUMNS,
    build_base_state,
    build_count_state,
    generate_strike_zone_grid,
    load_league_average_physics,
    simulate_plate_appearance,
)

artifacts = load_ev_artifacts(ROOT / "models")
physics = load_league_average_physics()

grid = generate_strike_zone_grid()
print(f"Grid locations: {len(grid):,}")
print(f"Competitive pitch types ({len(CORE_PITCH_TYPES)}): {CORE_PITCH_TYPES}")
print(f"Total scenarios per PA: {len(grid) * len(CORE_PITCH_TYPES):,}")
display(describe_ev_artifacts(artifacts))
display(physics.loc[physics["pitch_type"].isin(CORE_PITCH_TYPES)])


## 2. Simulate a plate appearance

Demo: **0-0 count**, **bases empty**, vs league-default right-handed batter profile.

In [ ]:
BATTER_LABEL = "Generic_R"
STAND = "R"
P_THROWS = "R"
COUNT_STATE = build_count_state(0, 0)
BASE_STATE = build_base_state("state_empty")
BATTER_PROFILE = league_default_batter_profile()

recommendations = simulate_plate_appearance(
    STAND,
    COUNT_STATE,
    BASE_STATE,
    BATTER_PROFILE,
    p_throws=P_THROWS,
    batter_id=BATTER_LABEL,
    artifacts=artifacts,
    league_physics=physics,
)

print(f"Scored {len(recommendations):,} pitch/location combinations\n")
display(recommendations.head(5).round({
    "plate_x": 2,
    "plate_z": 2,
    "Swing Prob": 3,
    "Whiff Prob": 3,
    "xwobacon": 3,
    "ERV": 4,
}))


## 3. ERV heatmap — best pitch type

Visualize ERV across the strike zone for the #1 recommended pitch type.

In [ ]:
best_type = recommendations.iloc[0]["Type"]
subset = recommendations.loc[recommendations["Type"] == best_type]

pivot = subset.pivot(index="plate_z", columns="plate_x", values="ERV")
pivot = pivot.sort_index(ascending=False)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(
    pivot.values,
    aspect="auto",
    origin="upper",
    extent=[
        pivot.columns.min() - 0.125,
        pivot.columns.max() + 0.125,
        pivot.index.max() + 0.125,
        pivot.index.min() - 0.125,
    ],
    cmap="RdYlGn_r",
)
ax.set_title(f"ERV heatmap — {best_type} vs {BATTER_LABEL} (0-0, bases empty)")
ax.set_xlabel("plate_x (ft)")
ax.set_ylabel("plate_z (ft)")
fig.colorbar(im, ax=ax, label="ERV (lower = better for pitcher)")
fig.tight_layout()
plt.show()


## 4. Compare situations

Same batter profile, **0-2 count** — recommendations should shift toward chase/waste and higher whiff probability.

In [ ]:
two_strike = simulate_plate_appearance(
    STAND,
    build_count_state(0, 2),
    BASE_STATE,
    BATTER_PROFILE,
    p_throws=P_THROWS,
    batter_id=BATTER_LABEL,
    artifacts=artifacts,
    league_physics=physics,
)

compare = pd.concat(
    [
        recommendations.head(3).assign(count="0-0"),
        two_strike.head(3).assign(count="0-2"),
    ],
    ignore_index=True,
)

display(compare.round(4))


## 5. Summary

The AI Catcher cross-joins a **221-point location grid** with **10 competitive pitch types**, applies league-average physics, injects a **rolling batter profile**, and ranks every combination by spatially smoothed **ERV** from the RF chain.

- **Best pitches** minimize `expected_run_value` (pitcher-favorable)
- **Swing Prob / Whiff Prob / xwOBAcon** decompose why a location works
- Customize `BATTER_PROFILE` (rates + bat-tracking) and any `count_*` / `state_*` one-hot to personalize recommendations